# M4.14 · Encoder-decoder Transformer (Attention Is All You Need)

Vaswani ve ark. (2017)'nin özgün kodlayıcı–kod çözücü Transformer'ı: N kodlayıcı katmanı (öz-attention + FFN, artık + Post-LN), N kod çözücü katmanı (maskeli öz-attention, kodlayıcı çıktısına cross-attention, FFN), sinüzoidal konum kodlaması ve göreli konum özelliği, √d ile ölçeklenen paylaşılan embedding'ler, dolgu ve nedensel maskeler, etiket yumuşatma ve Noam öğrenme oranı takvimi. Base modelin parametre sayımı elle, NumPy'da maskeli cross-attention, PyTorch'ta nn.TransformerEncoder- /DecoderLayer ile ağırlık düzeyinde aynı katmanlar ve tarih çevirisinde eğitim (embedding başlatma ölçeğinin neden kritik olduğu dahil), sonda kodlayıcı belleğini yeniden kullanan çözümleme. Ekler: RoPE, nedensel maskenin paralel eğitimi mümkün kılması, maske ailesi ve çok katmanlı erişim.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-14-transformer-ed.html](https://m-gonul.github.io/pytorch-lab/m4-14-transformer-ed.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_14_transformer_ed.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t():\n        P = need(ns, \"positional_encoding\")(64, 512)\n        close(P[1][0], math.sin(1), what=\"P[1,0]\")\n        close(P[2][1], math.cos(2), what=\"P[2,1]\")\n        close(P[22][100], P[60][100], atol=0.05, what=\"38 konum arayla boyut 100 neredeyse aynı (Géron)\")\n        close(P[22][101], P[60][101], atol=0.05, what=\"boyut 101\")\n        ensure(len(set(tuple(round(v, 6) for v in row) for row in P)) == 64, \"her konum benzersiz\")\n        close(need(ns, \"noam_lr\")(4000, 512, 4000), 512 ** -0.5 * 4000 ** -0.5, what=\"Noam tepe\")\n        ensure(ns[\"noam_lr\"](2000, 512, 4000) < ns[\"noam_lr\"](4000, 512, 4000) > ns[\"noam_lr\"](16000, 512, 4000), \"ısınma sonra azalma\")\n        close(ns[\"noam_lr\"](16000, 512, 4000) / ns[\"noam_lr\"](4000, 512, 4000), 0.5, what=\"4× adımda yarıya (1/√s)\")\n        close(need(ns, \"smooth_targets\")(1, 5, 0.1), [0.02, 0.92, 0.02, 0.02, 0.02], what=\"hedef\")\n        q = ns[\"smooth_targets\"](1, 5, 0.1)\n        close(need(ns, \"smoothed_ce\")([math.log(x) for x in q], 1, 0.1), -sum(x * math.log(x) for x in q), what=\"en iyi tahminde CE = H(q)\")\n        p = need(ns, \"transformer_params\")(512, 8, 2048, 6, 37000)\n        ensure(p == {\"enc_layer\": 3152384, \"dec_layer\": 4204032, \"total\": 63082496}, f\"base {p}\")\n        ensure(need(ns, \"causal_allowed\")(4) == 10, \"nedensel\")\n        rows = []\n        for name, (d, h, ff, N) in ((\"base\", (512, 8, 2048, 6)), (\"big\", (1024, 16, 4096, 6)), (\"d aşamasındaki\", (64, 4, 128, 2))):\n            pp = ns[\"transformer_params\"](d, h, ff, N, 37000 if name != \"d aşamasındaki\" else 24)\n            rows.append([name, d, N, f\"{pp['total'] / 1e6:.2f} M\", f\"{37000 * d / pp['total']:.0%}\" if name != \"d aşamasındaki\" else \"—\"])\n        R_.table(\"Parametre sayımları\", [\"model\", \"d\", \"N\", \"toplam\", \"embedding payı\"], rows,\n                 note=\"Base modelde parametrelerin ~%30'u paylaşılan embedding'dedir; paylaşım olmasaydı (kaynak, hedef, çıktı ayrı) 37M daha eklenirdi. big (213M makalede) d'yi iki katına çıkarır: katman parametreleri ~4×.\")\n    R_.test(\"formüller\", t)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"pe\", \"offset_rotation\", \"mha\", \"decoder_masks\", \"ls_ce\")])\n    rng = np.random.default_rng(0)\n\n    def t_pe():\n        P = need(ns, \"pe\")(100, 16)\n        close(P[3, 4], np.sin(3 / 10000 ** (4 / 16)), what=\"PE\")\n        for k in (1, 5, 17):\n            R = need(ns, \"offset_rotation\")(k, 16)\n            close(P[:100 - k] @ R.T, P[k:], atol=1e-10, what=f\"P[p + {k}] = R_k P[p]\")\n            close(R @ R.T, np.eye(16), atol=1e-12, what=\"R ortogonal\")\n        G = P @ P.T\n        close(G[10, 15], G[40, 45], atol=1e-10, what=\"P_p · P_{p+k} yalnız k'ya bağlı\")\n    R_.test(\"sinüzoidal kodlama\", t_pe)\n\n    def t_mha():\n        B, T, S, d, h = 2, 4, 5, 8, 2\n        xq, xkv = rng.normal(size=(B, T, d)), rng.normal(size=(B, S, d))\n        W = [rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(4)]\n        sm, cm = need(ns, \"decoder_masks\")([4, 2], [5, 3], T, S)\n        ensure(sm.shape == (2, 4, 4) and cm.shape == (2, 4, 5), \"şekiller\")\n        ensure(sm[0].sum() == 10 and not sm[1, 3, 2] and sm[1, 3, 1] and cm[1, 0].tolist() == [True] * 3 + [False] * 2, \"maskeler\")\n        out = need(ns, \"mha\")(xq, xkv, *W, h, cm)\n        for b in range(B):\n            heads = []\n            for j in range(h):\n                sl = slice(j * d // h, (j + 1) * d // h)\n                q, k, v = (xq[b] @ W[0])[:, sl], (xkv[b] @ W[1])[:, sl], (xkv[b] @ W[2])[:, sl]\n                L = [5, 3][b]\n                s = q @ k[:L].T / np.sqrt(d // h)\n                a = np.exp(s - s.max(1, keepdims=True))\n                a /= a.sum(1, keepdims=True)\n                heads.append(a @ v[:L])\n            close(out[b], np.concatenate(heads, 1) @ W[3], atol=1e-10, what=\"cross-attention (dolgu anahtarları dışarıda)\")\n        xkv2 = xkv.copy()\n        xkv2[1, 3:] += 100\n        close(ns[\"mha\"](xq, xkv2, *W, h, cm)[1], out[1], atol=1e-10, what=\"dolgu içeriği çıktıyı etkilemez\")\n    R_.test(\"maskeli multi-head cross-attention\", t_mha)\n\n    def t_ls():\n        logits = rng.normal(size=(2, 3, 5))\n        t = np.array([[1, 4, 0], [2, 0, 0]])\n        lp = logits - np.log(np.exp(logits).sum(-1, keepdims=True))\n        ref = []\n        for (b, i) in ((0, 0), (0, 1), (1, 0)):\n            q = np.full(5, 0.02)\n            q[t[b, i]] = 0.92\n            ref.append(-(q * lp[b, i]).sum())\n        close(need(ns, \"ls_ce\")(logits, t, 0.1, 0), np.mean(ref), what=\"yumuşatılmış CE\")\n        close(ns[\"ls_ce\"](logits, t, 0.0, 0), -np.mean([lp[0, 0, 1], lp[0, 1, 4], lp[1, 0, 2]]), what=\"ε = 0 → normal CE\")\n    R_.test(\"etiket yumuşatma\", t_ls)\n",
 "d": "import time\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t_layers():\n        torch.manual_seed(0)\n        d, h, ff = 16, 4, 32\n        ref = torch.nn.TransformerEncoderLayer(d, h, ff, dropout=0.0, batch_first=True)\n        mine = need(ns, \"EncoderLayer\")(d, h, ff, 0.0)\n        mine.load_state_dict(ref.state_dict())\n        x = torch.randn(3, 7, d, requires_grad=True)\n        pad = torch.zeros(3, 7, dtype=torch.bool)\n        pad[1, 5:] = True\n        pad[2, 2:] = True\n        a, b = ref(x, src_key_padding_mask=pad), mine(x, pad)\n        close(a[~pad].detach().numpy(), b[~pad].detach().numpy(), atol=1e-5, what=\"EncoderLayer = nn.TransformerEncoderLayer\")\n        refd = torch.nn.TransformerDecoderLayer(d, h, ff, dropout=0.0, batch_first=True)\n        md = need(ns, \"DecoderLayer\")(d, h, ff, 0.0)\n        md.load_state_dict(refd.state_dict())\n        y = torch.randn(3, 5, d, requires_grad=True)\n        causal = torch.triu(torch.full((5, 5), float(\"-inf\")), 1)\n        close(refd(y, x, tgt_mask=causal, memory_key_padding_mask=pad).detach().numpy(), md(y, x, causal, pad).detach().numpy(), atol=1e-5, what=\"DecoderLayer = nn.TransformerDecoderLayer\")\n        y2 = y.detach().clone()\n        y2[:, 3:] += 10\n        ensure(torch.allclose(md(y2, x, causal, pad)[:, :3], md(y.detach(), x, causal, pad)[:, :3], atol=1e-5), \"nedensel: gelecek token'lar geçmiş çıktıları etkilemez\")\n    R_.test(\"katmanlar PyTorch ile birebir\", t_layers)\n\n    def t_train():\n        X, Y = need(ns, \"dates\")(12000, 0)\n        Xt, Yt = ns[\"dates\"](1000, 1)\n        x, y = ns[\"encode\"](X, Y)\n        xt, yt = ns[\"encode\"](Xt, Yt)\n        yin = torch.cat([torch.full((len(y), 1), ns[\"SOS\"]), y[:, :-1]], 1)\n        rows, acc = [], {}\n        for name, std in ((\"embedding N(0, 1/d) (doğru)\", None), (\"embedding N(0, 1) × √d\", 1.0)):\n            torch.manual_seed(0)\n            m = need(ns, \"Transformer\")(64, 4, 128, 2, 0.1, std)\n            ensure(tuple(m(x[:3], yin[:3]).shape) == (3, 11, ns[\"V_OUT\"]), \"logit şekli\")\n            opt = torch.optim.Adam(m.parameters(), 1.0, betas=(0.9, 0.98), eps=1e-9)\n            sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 2 * 64 ** -0.5 * min((s + 1) ** -0.5, (s + 1) * 400 ** -1.5))\n            t0 = time.perf_counter()\n            for ep in range(2):\n                m.train()\n                perm = torch.randperm(len(x))\n                for i in range(0, len(x), 64):\n                    idx = perm[i:i + 64]\n                    loss = F.cross_entropy(m(x[idx], yin[idx]).flatten(0, 1), y[idx].flatten(), label_smoothing=0.1)\n                    opt.zero_grad()\n                    loss.backward()\n                    opt.step()\n                    sch.step()\n            m.eval()\n            p = m.greedy(xt)\n            acc[name] = (p == yt).all(1).float().mean().item()\n            with torch.no_grad():\n                e = (m.src.weight * 8).norm(dim=1).mean().item()\n            rows.append([name, f\"{e:.1f}\", f\"{(p == yt).float().mean().item():.3f}\", f\"{acc[name]:.3f}\", f\"{time.perf_counter() - t0:.0f} sn\"])\n        a, b = acc.values()\n        ensure(a > 0.5 and a > b + 0.3, f\"doğruluklar {acc}\")  # CPU/donanım farkına dayanıklı eşik\n        R_.table(\"Tarih çevirisi: Transformer d = 64, N = 2 + 2, Noam (w = 400), ε_ls = 0.1, 2 epoch\", [\"başlatma\", \"‖emb·√d‖ (eğitim sonu)\", \"token doğr.\", \"tam eşleşme\", \"süre\"], rows,\n                 note=\"Sinüzoidal PE'nin her bileşeni [−1, 1]'de, vektör normu √(d/2) ≈ 5.7. N(0, 1) embedding × √d = 8 normu ~64 olur: konum sinyali gürültü düzeyinde kalır, model karakterlerin sırasını çözemez. Paylaşılan embedding'de (makale) başlatma ~N(0, 1/d) olduğu için √d çarpanı onu PE ile aynı ölçeğe getirir.\")\n    R_.test(\"eğitim\", t_train)\n",
 "e": "import math\nimport time\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom labcheck import need, ensure, fmt_time\n\n\nclass _TF(nn.Module):\n    def __init__(self, V=50, d=128, N=4):\n        super().__init__()\n        self.emb = nn.Embedding(V, d)\n        self.core = nn.Transformer(d, 4, N, N, 4 * d, dropout=0.0, batch_first=True)\n        self.out = nn.Linear(d, V)\n\n    def encode(self, x):\n        return self.core.encoder(self.emb(x)), None\n\n    def decode(self, y, mem, pad):\n        T = y.shape[1]\n        return self.out(self.core.decoder(self.emb(y), mem, tgt_mask=torch.triu(torch.full((T, T), float(\"-inf\")), 1)))\n\n    def forward(self, x, y):\n        mem, pad = self.encode(x)\n        return self.decode(y, mem, pad)\n\n\ndef check(ns, R_):\n    def t():\n        torch.manual_seed(0)\n        m = _TF().eval()\n        rows = []\n        for S, T in ((16, 8), (128, 16), (512, 32)):\n            x = torch.randint(2, 50, (16, S))\n            t0 = time.perf_counter()\n            a = need(ns, \"greedy_naive\")(m, x, T)\n            ta = time.perf_counter() - t0\n            t0 = time.perf_counter()\n            b = need(ns, \"greedy_cached\")(m, x, T)\n            tb = time.perf_counter() - t0\n            ensure(torch.equal(a, b) and tuple(a.shape) == (16, T), \"aynı çıktı\")\n            rows.append([S, T, fmt_time(ta), fmt_time(tb), f\"{ta / tb:.1f}×\"])\n        R_.table(\"Açgözlü çözümleme, batch 16, d = 128, 4 + 4 katman (CPU)\", [\"kaynak uzunluğu S\", \"üretilen T\", \"kodlayıcı her adımda\", \"bellek bir kez\", \"hızlanma\"], rows,\n                 note=\"Kodlayıcı çıktısı çözümleme boyunca değişmez: bir kez hesaplanıp N kod çözücü katmanına verilir. Yeniden hesaplamak T × kodlayıcı maliyeti (S² attention) ekler; kaynak uzadıkça fark büyür. Ek olarak cross-attention'ın K, V izdüşümleri de önbelleğe alınabilir, öz-attention için KV-cache (M4.7) kod çözücü tarafını hızlandırır.\")\n    R_.test(\"kodlayıcı belleğini yeniden kullanma\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: max_loops(src, 0, func=\"rope\"))\n\n    def t():\n        rng = np.random.default_rng(0)\n        d = 32\n        q, k = rng.normal(size=(1, d)), rng.normal(size=(1, d))\n        r = need(ns, \"rope\")\n        close(np.linalg.norm(r(q, [7])), np.linalg.norm(q), what=\"normu korur\")\n        x = np.array([[1.0, 0.0]])\n        close(r(x, [1], 10000), np.array([[np.cos(1), np.sin(1)]]), what=\"2B döndürme\")\n        rows = []\n        for m_, n_ in ((3, 1), (50, 48), (1000, 998), (5, 7), (1005, 1007)):\n            sr = float((r(q, [m_]) @ r(k, [n_]).T)[0, 0])\n            P = np.zeros((2000, d))\n            a = np.arange(2000)[:, None] * 10000.0 ** (-np.arange(0, d, 2) / d)\n            P[:, 0::2], P[:, 1::2] = np.sin(a), np.cos(a)\n            sa = float(((q + P[m_]) @ (k + P[n_]).T)[0, 0])\n            rows.append([m_, n_, m_ - n_, f\"{sr:+.4f}\", f\"{sa:+.4f}\"])\n        ensure(abs(float(rows[0][3]) - float(rows[1][3])) < 1e-3 and abs(float(rows[1][3]) - float(rows[2][3])) < 1e-3 and abs(float(rows[3][3]) - float(rows[4][3])) < 1e-3, \"RoPE skoru yalnız m − n'ye bağlı\")\n        R_.table(\"q·k skoru: RoPE vs toplamsal sinüzoidal PE\", [\"m\", \"n\", \"m − n\", \"RoPE\", \"toplamsal PE\"], rows,\n                 note=\"RoPE sorgu ve anahtarı konumlarına göre döndürür: ⟨R_m q, R_n k⟩ = ⟨q, R_{n−m} k⟩ — skor yalnız göreli uzaklığa bağlıdır, mutlak konum skora hiç girmez. Toplamsal PE'de (q + P_m)·(k + P_n) çapraz terimler mutlak konuma bağlıdır. LLaMA ve çoğu modern LLM RoPE kullanır; değerlere (V) uygulanmaz.\")\n    R_.test(\"RoPE\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        T, d = 9, 8\n        X = rng.normal(size=(T, d))\n        W = [rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3)]\n        full = need(ns, \"decoder_full\")(X, *W, True)\n        inc = need(ns, \"decoder_incremental\")(X, *W)\n        close(full, inc, atol=1e-12, what=\"nedensel tam geçiş = adım adım çözümleme\")\n        nomask = ns[\"decoder_full\"](X, *W, False)\n        diff = np.abs(nomask - inc).max(1)\n        ensure(diff[-1] < 1e-12 and diff[0] > 1e-3, \"maskesiz: yalnız son konum eşleşir\")\n        rows = [[t, f\"{diff[t]:.2e}\"] for t in range(T)]\n        R_.table(\"Maskesiz tam geçiş ile adım adım çözümleme arasındaki fark\", [\"konum t\", \"max |fark|\"], rows,\n                 note=\"Nedensel maskeyle T konumun tamamı tek bir matris çarpımıyla hesaplanır ve her konum, çıkarımda göreceği şeyin aynısını görür: öğretmen zorlaması paralel olur (RNN'de T sıralı adım). Maske olmasaydı eğitimde t. konum geleceği görürdü — 'hile' — ve çıkarımla tutarsız olurdu; tek istisna her şeyi zaten gören son konumdur.\")\n    R_.test(\"paralel eğitim\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        ensure(need(ns, \"causal\")(4).sum() == 10, \"nedensel\")\n        sw = need(ns, \"sliding_window\")(8, 3)\n        ensure(sw[5].tolist() == [False, False, False, True, True, True, False, False], \"kayan pencere\")\n        pl = need(ns, \"prefix_lm\")(6, 3)\n        ensure(pl[0, 2] and not pl[3, 4] and pl[4, 1], \"prefix-LM\")\n        st = need(ns, \"strided\")(12, 4)\n        ensure(st[9, 8] and st[9, 5] and st[9, 1] and not st[9, 4], \"strided (Sparse Transformer)\")\n        T = 256\n        rows = []\n        for name, M in ((\"nedensel\", ns[\"causal\"](T)), (\"kayan pencere w = 8\", ns[\"sliding_window\"](T, 8)), (\"strided s = 16\", ns[\"strided\"](T, 16))):\n            r = [name, f\"{M.sum() / ns['causal'](T).sum():.0%}\"]\n            for L in (1, 2, 4, 8):\n                R = need(ns, \"reach\")(M, L)\n                r.append(int(R[T - 1].sum()))\n            rows.append(r)\n        Rw = ns[\"reach\"](ns[\"sliding_window\"](T, 8), 4)\n        ensure(int(Rw[T - 1].sum()) == 4 * 7 + 1, \"w = 8, 4 katman: 4·(w − 1) + 1 = 29 token\")\n        ensure(int(ns[\"reach\"](ns[\"strided\"](T, 16), 2)[T - 1].sum()) == T, \"strided: 2 katmanda tüm geçmiş\")\n        R_.table(\"T = 256: son token'ın L katman sonra 'görebildiği' girdi sayısı\", [\"maske\", \"nedensele göre çift\", \"L = 1\", \"L = 2\", \"L = 4\", \"L = 8\"], rows,\n                 note=\"Seyrek maskeler attention maliyetini düşürür ama bilgi katmanlar boyunca yayılarak yine uzağa ulaşır: kayan pencerede erişim L·(w − 1) + 1 (CNN'in receptive field'ı gibi, Longformer/Mistral), strided desende (Child ve ark. 2019) iki katmanda tüm geçmiş — O(T√T) maliyetle.\")\n    R_.test(\"maske ailesi\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_14_transformer_ed as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Mimari** (Vaswani ve ark. 2017): sol taraf **kodlayıcı** — kelime ID'lerinden oluşan `[batch, kaynak uzunluğu]` girdiyi `[batch, kaynak uzunluğu, 512]` gösterimlere çevirir; üst kısmı N = 6 kez yığılır. Sağ taraf **kod çözücü** — eğitimde hedef cümleyi (bir sağa kaydırılmış) girdi olarak alır, kodlayıcının **son** çıktısı N kod çözücü katmanının her birine verilir; her adımda sözlük üzerinde olasılık üretir: `[batch, hedef uzunluğu, sözlük]`. Çıkarımda önceki çıktılar geri beslenir (SOS'tan EOS'a).

Her alt katman: artık bağlantı + LayerNorm, $\mathrm{LN}(x+\mathrm{Sublayer}(x))$ (Post-LN). Üç attention türü: kodlayıcının **öz-attention**'ı her kelimenin aynı cümledeki diğer kelimelerle ilişkisini kodlar (dolgu anahtarları maskeli); kod çözücünün **maskeli** öz-attention'ı yalnız öncesine bakar; kod çözücünün üst attention'ı (**cross-attention**) girdi cümlesine bakar — Q kod çözücüden, K ve V kodlayıcıdan. FFN: $\max(0,xW_1+b_1)W_2+b_2$, $d_{ff}=4d$.

**Konum kodlaması:** attention sırayı görmez; diğer katmanlar zamanca dağıtılmış. p. konumdaki vektör p. kelimenin embedding'ine eklenir. Öğrenilebilir de olabilir ama makale sabit sinüzoidal seçer: aynı başarım, keyfi uzun cümlelere uzanır, her konuma benzersiz; ayrıca $P_{p+k}$, $P_p$'nin doğrusal bir fonksiyonudur (her boyut çifti bir döndürme) — göreli konum öğrenmeyi kolaylaştırır. Örneğin boyut 100 ve 101'in dalga boyu ≈ 38 konum: 38 kelime arayla bu iki boyut aynıdır. Embedding'ler $\sqrt d$ ile çarpılır ve kaynak/hedef embedding'leri ile çıktı katmanı tek bir matrisi paylaşır.

**Eğitim:** Adam (β₂ = 0.98, ε = 10⁻⁹), **Noam takvimi** $\mathrm{lr}=d^{-0.5}\min(s^{-0.5},s\,w^{-1.5})$ — w = 4000 adım doğrusal ısınma, sonra $1/\sqrt s$ azalma; dropout 0.1; **etiket yumuşatma** ε = 0.1 (perpleksiteyi kötüleştirir, doğruluğu/BLEU'yu iyileştirir).

## 1 · Elle

<!-- çözüm -->

$P_{1,0}=\sin(1/10000^0)=\sin1=0.841$; $P_{2,1}=\cos(2)=-0.416$. Boyut 100 (i = 50): açısal frekans $\omega=10000^{-100/512}$, dalga boyu $2\pi/\omega=2\pi\cdot10000^{0.1953}=37.96$ ≈ 38.

Kodlayıcı katmanı: MHA $4(d^2+d)=1\,050\,624$; FFN $d\,d_{ff}+d_{ff}+d_{ff}\,d+d=2\,099\,712$; iki LN $2\cdot2d=2\,048$ → **3 152 384**. Kod çözücü: iki MHA + FFN + üç LN → **4 204 032**. Toplam: $6\cdot3\,152\,384+6\cdot4\,204\,032+37\,000\cdot512=63\,082\,496$ (makale ≈ 65M; fark sözlük boyu ve sayım ayrıntıları).

Noam tepesi s = w'de: $512^{-0.5}\cdot4000^{-0.5}=1/\sqrt{2\,048\,000}=6.99\cdot10^{-4}$.

Etiket yumuşatma (PyTorch): $q=(1-\varepsilon)\,\text{onehot}+\varepsilon/V$ → doğru sınıf $0.9+0.02=0.92$, diğerleri 0.02. CE $\ge H(q)=-(0.92\ln0.92+4\cdot0.02\ln0.02)=0.390$: model hiçbir zaman olasılık 1 vermeye zorlanmaz — logit'ler sınırlı kalır.

Nedensel maske T = 4: alt üçgen 1 + 2 + 3 + 4 = 10.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Sinüzoidal konum kodlaması, d = 512: $P_{p,2i}=\sin(p/10000^{2i/d})$, $P_{p,2i+1}=\cos(\cdot)$**

- `pe1`: $P_{1,0}$
- `pe2`: $P_{2,1}$
- `wl`: Boyut çifti (100, 101)'in dalga boyu (konum cinsinden)

**Base Transformer: d = 512, h = 8, d_ff = 2048 (bias ve LN dahil)**

- `enc`: Bir kodlayıcı katmanının parametresi
- `dec`: Bir kod çözücü katmanının parametresi
- `tot`: N = 6, paylaşılan 37 000 token'lık embedding (çıktı katmanı dahil, bias yok): toplam

**Eğitim**

- `noam`: Noam: $d^{-0.5}\min(s^{-0.5}, s\,w^{-1.5})$, w = 4000 — tepe öğrenme oranı
- `lsq`: Etiket yumuşatma ε = 0.1, V = 5 (PyTorch tanımı): doğru sınıfın hedef olasılığı
- `lsh`: Aynı hedefle ulaşılabilecek en küçük çapraz entropi (nat)
- `causal`: T = 4 nedensel maskede izin verilen (sorgu, anahtar) çifti

In [ ]:
cevaplar = dict(
    pe1=None,
    pe2=None,
    wl=None,
    enc=None,
    dec=None,
    tot=None,
    noam=None,
    lsq=None,
    lsh=None,
    causal=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"pe1":[0.8414709848078965,0.000001],"pe2":[-0.4161468365471424,0.000001],"wl":[37.969062003258976,0.0001],"enc":[3152384,0],"dec":[4204032,0],"tot":[63082496,0],"noam":[0.0006987712429686843,1e-9],"lsq":[0.92,1e-9],"lsh":[0.3896729206581786,0.000001],"causal":[10,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `positional_encoding(n, d)` → `n × d` liste.
- `noam_lr(step, d, warmup, factor=1.0)` (step ≥ 1).
- `smooth_targets(t, V, eps)` — $q=(1-\varepsilon)\,\text{onehot}(t)+\varepsilon/V$.
- `smoothed_ce(logp, t, eps)` — `logp` log olasılık listesi.
- `transformer_params(d, h, dff, N, V)` → `{"enc_layer", "dec_layer", "total"}` (paylaşılan embedding, çıktı katmanında bias yok, Post-LN).
- `causal_allowed(T)` — nedensel maskede izinli çift sayısı.

In [ ]:
%%writefile stage_b.py
import math


def positional_encoding(n, d):
    ...


def noam_lr(step, d, warmup, factor=1.0):
    ...


def smooth_targets(t, V, eps):
    ...


def smoothed_ce(logp, t, eps):
    ...


def transformer_params(d, h, dff, N, V):
    ...


def causal_allowed(T):
    ...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz.

- `pe(n, d)`.
- `offset_rotation(k, d)` → `(d, d)` matris $R_k$: tüm p için $P_{p+k}=R_kP_p$ (her (sin, cos) çifti için 2×2 döndürme).
- `mha(xq, xkv, Wq, Wk, Wv, Wo, h, allow)` — `xq (B, T, d)`, `xkv (B, S, d)`, ağırlıklar `(d, d)`, `allow (B, T, S)` bool (True = izinli) → `(B, T, d)`.
- `decoder_masks(tgt_lens, src_lens, T, S)` → `(self (B, T, T), cross (B, T, S))`: öz-maske = nedensel ∧ hedef anahtar dolgu değil; cross = kaynak anahtar dolgu değil.
- `ls_ce(logits, t, eps, pad=0)` — dolgu hariç ortalama yumuşatılmış CE.

In [ ]:
%%writefile stage_c.py
import numpy as np


def pe(n, d):
    ...


def offset_rotation(k, d):
    ...


def mha(xq, xkv, Wq, Wk, Wv, Wo, h, allow):
    ...


def decoder_masks(tgt_lens, src_lens, T, S):
    ...


def ls_ce(logits, t, eps, pad=0):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. Görev: M4.13'teki tarih çevirisi (`dates`, `encode` verilir).

- `EncoderLayer(d, h, ff, dropout)` ve `DecoderLayer(d, h, ff, dropout)` — Post-LN, ReLU; **öznitelik adları PyTorch'unkilerle aynı** (`self_attn`, `multihead_attn`, `linear1`, `linear2`, `norm1..3`; `nn.MultiheadAttention(batch_first=True)` kullan) ki `nn.TransformerEncoderLayer`/`DecoderLayer` state_dict'i doğrudan yüklensin. `EncoderLayer.forward(x, pad)`; `DecoderLayer.forward(y, mem, causal, mem_pad)`.
- `Transformer(d=64, h=4, ff=128, N=2, dropout=0.1, emb_std=None)` — kaynak/hedef embedding'leri ($\times\sqrt d$) + sinüzoidal PE, N + N katman, çıktı doğrusal katmanı. `emb_std=None` → $d^{-1/2}$; bir sayı verilirse embedding'ler $N(0, \text{emb\_std}^2)$.
  - `encode(x)` → `(bellek, pad)`; `decode(yin, mem, pad)` → logit'ler; `forward(x, yin)`; `greedy(x, T=11)`.

In [ ]:
%%writefile stage_d.py
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
IN = sorted(set("".join(MONTHS) + "0123456789, "))
OUT = list("0123456789-")
PAD, SOS, EOS = 0, 1, 2
V_IN, V_OUT = len(IN) + 1, len(OUT) + 3


def dates(n, seed):
    r = random.Random(seed)
    X, Y = [], []
    for _ in range(n):
        y, m, d = r.randint(1000, 2999), r.randint(1, 12), r.randint(1, 28)
        X.append([f"{MONTHS[m - 1]} {d}, {y}", f"{d} {MONTHS[m - 1]} {y}", f"{MONTHS[m - 1][:3]} {d:02d} {y}"][r.randint(0, 2)])
        Y.append(f"{y}-{m:02d}-{d:02d}")
    return X, Y


def encode(X, Y):
    iv = {c: i + 1 for i, c in enumerate(IN)}
    ov = {c: i + 3 for i, c in enumerate(OUT)}
    L = max(map(len, X))
    xs = torch.tensor([[iv[c] for c in x] + [PAD] * (L - len(x)) for x in X])
    ys = torch.tensor([[ov[c] for c in y] + [EOS] for y in Y])
    return xs, ys


class EncoderLayer(nn.Module):
    def __init__(self, d, h, ff, dropout=0.0):
        super().__init__()
        ...

    def forward(self, x, pad):
        ...


class DecoderLayer(nn.Module):
    def __init__(self, d, h, ff, dropout=0.0):
        super().__init__()
        ...

    def forward(self, y, mem, causal, mem_pad):
        ...


class Transformer(nn.Module):
    def __init__(self, d=64, h=4, ff=128, N=2, dropout=0.1, emb_std=None):
        super().__init__()
        ...

    def encode(self, x):
        ...

    def decode(self, yin, mem, pad):
        ...

    def forward(self, x, yin):
        ...

    @torch.no_grad()
    def greedy(self, x, T=11):
        ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `greedy_naive(model, x, T)` — her adımda `model(x, y)` (kodlayıcı her adımda yeniden çalışır).
- `greedy_cached(model, x, T)` — kodlayıcıyı bir kez çalıştır, belleği yeniden kullan.
- Ölçüm: iki yöntemin süresi ve çıktı eşitliği; T ve kaynak uzunluğu büyürken.

In [ ]:
%%writefile stage_e.py
import time
import torch


def greedy_naive(model, x, T):
    ...


def greedy_cached(model, x, T):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · RoPE: göreli konum, döndürme ile

NumPy.

- `rope(x, pos, base=10000)` — `x (T, d)`, `pos (T,)`: her çift $(x_{2i},x_{2i+1})$'i $\theta_i=\text{pos}\cdot\text{base}^{-2i/d}$ açısıyla döndür. Döngüsüz.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def rope(x, pos, base=10000):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Nedensel maske = paralel öğretmen zorlaması

NumPy. Tek katmanlı, tek kafalı öz-attention "kod çözücü": $y=\mathrm{softmax}(QK^\top/\sqrt d+M)V$.

- `decoder_full(X, Wq, Wk, Wv, causal=True)` — tüm diziyi tek geçişte (nedensel maskeyle).
- `decoder_incremental(X, Wq, Wk, Wv)` — t = 0..T−1 için yalnız `X[:t+1]`'i görerek t. çıktıyı hesapla (döngü serbest).

In [ ]:
%%writefile extra_x2.py
import numpy as np


def decoder_full(X, Wq, Wk, Wv, causal=True):
    ...


def decoder_incremental(X, Wq, Wk, Wv):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Maske ailesi ve katmanlar boyunca erişim

NumPy, döngüsüz (katman döngüsü serbest).

- `causal(T)`, `sliding_window(T, w)` (nedensel, son w konum), `prefix_lm(T, P)` (ilk P konum kendi aralarında çift yönlü, sonrası nedensel), `strided(T, s)` (nedensel; son s konum ya da aradaki uzaklık s'nin katıysa izinli — Child ve ark.).
- `reach(allow, L)` — L katmandan sonra hangi girdinin hangi çıktıyı etkileyebildiği: $(A\lor I)^L$ boolean kuvveti.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def causal(T):
    ...


def sliding_window(T, w):
    ...


def prefix_lm(T, P):
    ...


def strided(T, s):
    ...


def reach(allow, L):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Vaswani ve ark. (2017): küçük bir çeviri modeli

[Vaswani ve ark. — Attention Is All You Need (NeurIPS 2017)](https://arxiv.org/abs/1706.03762)

d aşamasındaki modeli Multi30k (de→en, ~30 bin cümle) ya da Tatoeba en–tr üzerinde BPE ile eğit: paylaşılan kaynak/hedef sözlük ve embedding–çıktı ağırlık bağlama, Noam takvimi, etiket yumuşatma 0.1, dropout 0.1, beam search (k = 4, α = 0.6, M4.13). Makale Tablo 3'ün küçük ablasyonunu yap: (A) head sayısı 1/4/8, (B) d_k küçültme, (D) dropout ve ε_ls = 0, (E) öğrenilmiş konum embedding'i — BLEU ile raporla. M4.13'teki RNN + attention modeliyle aynı veride karşılaştır.


**Adımlar**

1. Kontrol noktası ortalaması (son 5) ve çözümlemede uzunluk cezası.

**Kabul ölçütleri**

- Transformer, RNN + attention modelinden daha yüksek BLEU'ya daha kısa sürede ulaşır.
- Etiket yumuşatma perpleksiteyi kötüleştirip BLEU'yu iyileştirir (makaledeki gözlem).

## Makale ödevi: Xiong ve ark. (2020): Post-LN vs Pre-LN

[Xiong ve ark. — On Layer Normalization in the Transformer Architecture (ICML 2020)](https://arxiv.org/abs/2002.04745)

Makalenin kuramsal sonucunu (Post-LN'de son katmanların gradyanı başlangıçta büyük → warm-up şart) deneyle göster: d aşamasındaki katmanların Pre-LN sürümünü yaz; 6 ve 12 katmanlı modellerde başlangıçta katman başına gradyan normlarını çiz (Şekil 3), warm-up'lı/warm-up'sız ve farklı tepe öğrenme oranlarında eğitim eğrilerini karşılaştır.


**Adımlar**

1. Pre-LN: $x+\mathrm{Sublayer}(\mathrm{LN}(x))$ ve en sonda ek bir LN.

**Kabul ölçütleri**

- Post-LN'de gradyan normu çıkışa doğru artar; Pre-LN'de yaklaşık sabittir.
- Pre-LN warm-up olmadan eğitilebilir; Post-LN büyük öğrenme oranında warm-up'sız ıraksar.

## Kaynaklar

- [Vaswani ve ark. — Attention Is All You Need (NeurIPS 2017)](https://arxiv.org/abs/1706.03762) — arXiv
- [Rush ve ark. — The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — MIT
- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 16](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Su ve ark. — RoFormer: Rotary Position Embedding (2021)](https://arxiv.org/abs/2104.09864) — arXiv
- [Xiong ve ark. — On Layer Normalization in the Transformer Architecture (ICML 2020)](https://arxiv.org/abs/2002.04745) — arXiv
- [Müller, Kornblith, Hinton — When Does Label Smoothing Help? (NeurIPS 2019)](https://arxiv.org/abs/1906.02629) — arXiv
- [Beltagy, Peters, Cohan — Longformer (2020)](https://arxiv.org/abs/2004.05150) — arXiv
- [d2l.ai — 11.6 Self-Attention and Positional Encoding, 11.7 Transformer](https://d2l.ai/chapter_attention-mechanisms-and-transformers/transformer.html) — CC-BY-SA 4.0